Aquí importamos lo necesario

In [ ]:
import cv2
import numpy as np

**Tarea 2:** Cálculo del valor total de monedas

In [4]:
vid = cv2.VideoCapture(0)

# Diámetros oficiales en milímetros
diametro_monedas = {
    2.00: 25.75,
    1.00: 23.25,
    0.50: 24.25,
    0.20: 22.25,
    0.10: 19.75,
    0.05: 21.25,
    0.02: 18.75,
    0.01: 16.25
}

# Variables globales
moneda_seleccionada = None
diametro_ref = None
escala = None

def clasificar_moneda(diametro_medido, diccionario):
    moneda = min(
        diccionario,
        key=lambda valor: abs(diccionario[valor] - diametro_medido)
    )
    return moneda

def seleccionar_moneda(event, x, y, flags, param):
    global moneda_seleccionada, diametro_ref, escala

    if event == cv2.EVENT_LBUTTONDOWN:
        for c in contornos:
            if cv2.pointPolygonTest(c, (x, y), False) >= 0:
                # Medir la moneda seleccionada mediante el círculo mínimo
                centro, radio = cv2.minEnclosingCircle(c)
                diametro_ref = radio * 2

                # La moneda seleccionada se considera de 1 €
                escala = diametro_monedas[1.00] / diametro_ref
                moneda_seleccionada = (x, y)

                break

cv2.namedWindow("Vid")
cv2.setMouseCallback("Vid", seleccionar_moneda)

while True:
    ret, frame = vid.read()

    if ret:
        # Convertir a escala de grises
        img_gris = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

        # Suavizado para reducir ruido
        img_suavizada = cv2.GaussianBlur(
            img_gris, (5, 5), 0
        )

        # Umbral automático de Otsu
        _, img_th = cv2.threshold(
            img_suavizada,
            0,
            255,
            cv2.THRESH_BINARY_INV + cv2.THRESH_OTSU
        )

        # Detectar contornos externos
        contornos, _ = cv2.findContours(
            img_th,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        total = 0.0

        # Dibuja todos los contornos mientras no haya una escala
        if escala is None:
            cv2.drawContours(frame, contornos, -1, (0, 255, 0), 2)
        
        else:
            for c in contornos:
                centro, radio = cv2.minEnclosingCircle(c)
                diametro_px = radio * 2

                # Ignorar objetos demasiado pequeños y demasiado grandes
                if diametro_px < diametro_ref * 0.55:
                    continue

                if diametro_px > diametro_ref * 1.15:
                    continue

                # Solo dibuja los contornos de las monedas
                cv2.drawContours(frame, [c], -1, (0, 255, 0), 2)

                # Convertir píxeles a milímetros
                diametro_mm = diametro_px * escala

                # Clasificar por el diámetro oficial más cercano
                valor_moneda = clasificar_moneda(
                    diametro_mm,
                    diametro_monedas
                )

                # Dibujar el valor en el centro de la moneda
                x_centro = int(centro[0])
                y_centro = int(centro[1])

                texto = f"{valor_moneda:.2f} EUR"

                cv2.putText(
                    frame,
                    texto,
                    (x_centro - 45, y_centro),
                    cv2.FONT_HERSHEY_SIMPLEX,
                    0.6,
                    (0, 0, 255),
                    2,
                    cv2.LINE_AA
                )

                # Sumar el valor clasificado
                total += valor_moneda

            # Mostrar el total
            cv2.putText(
                frame,
                f"TOTAL = {total:.2f} EUR",
                (10, 40),
                cv2.FONT_HERSHEY_SIMPLEX,
                1,
                (0, 0, 255),
                2,
                cv2.LINE_AA
            )

        cv2.imshow("Vid", frame)
        cv2.imshow("Umbral", img_th)

    # ESC para salir
    if cv2.waitKey(20) == 27:
        break

vid.release()
cv2.destroyAllWindows()